In [ ]:
import os
from google.colab import drive

# 1. Ensure Drive is mounted
drive.mount('/content/drive')

# 2. Verify exact target path
target_path = "/content/drive/MyDrive/Colab Notebooks/ML Practice/Projects/Rice Classifier/rice_resnet18.pth"

if os.path.exists(target_path):
    print("SUCCESS: File found at exact path!")
else:
    print("File not found at target path. Searching entire Drive for '.pth' files...\n")
    # Search drive for any .pth files to find the true path
    found = False
    for root, dirs, files in os.walk("/content/drive/MyDrive"):
        for file in files:
            if file.endswith(".pth"):
                print(f"FOUND FILE AT: {os.path.join(root, file)}")
                found = True
    if not found:
        print("No .pth files were found anywhere in Google Drive.")

Mounted at /content/drive
SUCCESS: File found at exact path!


In [ ]:
from gradio.components import label
import gradio as gr
import torch
import torch.nn.functional as f
from PIL import Image
from torchvision import transforms
from torchvision.models import resnet18

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
class_names = ['Arborio', 'Basmati', 'Ipsala', 'Jasmine', 'Karacadag']

model = resnet18()
model.fc = torch.nn.Linear(model.fc.in_features, len(class_names))

model_path = "/content/drive/MyDrive/Colab Notebooks/ML Practice/Projects/Rice Classifier/rice_resnet18.pth"
model.load_state_dict(torch.load(model_path, map_location=device))
model = model.to(device)
model.eval()

data_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.RandomRotation(30),
    transforms.RandomHorizontalFlip()
])

def predict_image(input_image):
    if input_image.mode != "RGB":
        input_image = input_image.convert("RGB")

    tensor = data_transform(input_image).unsqueeze(0).to(device)
    with torch.no_grad():
        output = model(tensor)
        probs = f.softmax(output[0], dim = 0)

    return {class_names[i]: float(probs[i]) for i in range(len(class_names))}

demo = gr.Interface(
    fn = predict_image,
    inputs = gr.Image(type = 'pil', label = 'Upload Rice Grain Image'),
    outputs = gr.Label(num_top_classes = 5, label = "Prediction Probabilies"),
    title = "Rice Image Classifier",
    description = "Upload image of a rice grain"
)

demo.launch(share=True, debug = True)